### LOAD SILVER TABLES

In [0]:
from pyspark.sql.functions import *
claims = spark.table("vehicle_insurance.silver.claims")
customers = spark.table("vehicle_insurance.silver.customers")
policies = spark.table("vehicle_insurance.silver.policies")
vehicles = spark.table("vehicle_insurance.silver.vehicles")
payments = spark.table("vehicle_insurance.silver.payments")
fnol = spark.table("vehicle_insurance.silver.fnol")

### CREATE GOLD BASE TABLE

In [0]:
gold_clean = claims.alias("c") \
    .join(policies.alias("p"), "policy_id") \
    .join(customers.alias("cu"), col("p.customer_id") == col("cu.customer_id"), "left") \
    .join(vehicles.alias("v"), col("cu.customer_id") == col("v.customer_id"), "left") \
    .join(payments.alias("pay"), "claim_id", "left") \
    .join(fnol.alias("f"), "claim_id", "left") \
    .select(
        col("c.claim_id"),
        col("c.policy_id"),
        col("c.claim_amount"),
        col("c.claim_status"),
        col("c.claim_date"),
        
        col("cu.customer_id").alias("customer_id"),
        col("cu.name"),
        col("cu.city"),
        col("cu.state"),
        col("cu.risk_category"),
        
        col("p.premium_amount"),
        
        col("v.vehicle_type"),
        col("v.brand"),
        col("v.model"),
        
        col("pay.amount").alias("payment_amount"),
        
        col("f.event_time"),
        col("f.description")
    )

### 1. Total & Average Claim Amount + Settlement Rate

In [0]:
claims_kpi = gold_clean.agg(
    sum("claim_amount").alias("total_claim_amount"),
    avg("claim_amount").alias("avg_claim_amount")
)

display(claims_kpi)

### 2. Settlement Rate

In [0]:
claims_settlement_kpi = gold_clean.agg(
    (count(when(col("claim_status") == "SETTLED", True)) / count("*"))
    .alias("settlement_rate")
)

display(claims_settlement_kpi)

### 3. Average Payment per Claim

In [0]:
from pyspark.sql.functions import sum, countDistinct, col

agg_df = gold_clean.agg(
    sum("amount").alias("total_amount"),
    countDistinct("claim_id").alias("distinct_claims")
)

payment_kpi = agg_df.select(
    (col("total_amount") / col("distinct_claims")).alias("avg_payment_per_claim")
)

display(payment_kpi)

### 4. High Risk Customers

In [0]:
high_risk_customers = customers.filter(
    col("risk_category") == "HIGH"
).select("customer_id", "name", "risk_category")

display(high_risk_customers)

### 5. Policy Claim Ratio

In [0]:
policy_kpi = gold_df.groupBy("policy_id").agg(
    sum("claim_amount").alias("total_claim_amount"),
    first("premium_amount").alias("premium_amount")
).withColumn(
    "policy_claim_ratio",
    col("total_claim_amount") / col("premium_amount")
)

display(policy_kpi)

### 6. Average Claim Amount per Vehicle Type

In [0]:
vehicle_kpi = gold_df.groupBy("vehicle_type").agg(
    avg("claim_amount").alias("avg_claim_amount")
)

display(vehicle_kpi)

### 7. FNOL to Claim Conversion

In [0]:
fnol_kpi = fnol.agg(
    count("event_id").alias("total_fnol_events"),
    count("claim_id").alias("linked_claims")
).withColumn(
    "conversion_rate",
    col("linked_claims") / col("total_fnol_events")
)

display(fnol_kpi)

### 8. Incident Type Distribution

In [0]:
incident_kpi = fnol.groupBy("description").agg(
    count("*").alias("total_events")
)

display(incident_kpi)

### 9. Repeat Claims per Customer

In [0]:
repeat_claims = gold_df.groupBy(col("cu.customer_id")) \
    .agg(count("c.claim_id").alias("claim_count")) \
    .filter(col("claim_count") > 2)

display(repeat_claims)

### 10. Multiple FNOL per Claim

In [0]:
multiple_fnol = fnol.groupBy("claim_id").agg(
    count("event_id").alias("fnol_count")
).filter(
    col("fnol_count") > 1
)

display(multiple_fnol)

### 11. High Claim Outliers

In [0]:
high_claim_outliers = gold_df.filter(
    col("claim_amount") > 50000
).select(
    col("c.claim_id").alias("claim_id"),
    col("c.policy_id").alias("policy_id"),
    col("c.claim_amount").alias("claim_amount"),
    col("cu.customer_id").alias("customer_id")
)

display(high_claim_outliers)

### SAVING ALL THE GOLD TABLES

In [0]:
claims_kpi.write.format("delta").mode("overwrite").saveAsTable("vehicle_insurance.gold.claims_kpi")

claims_settlement_kpi.write.format("delta").mode("overwrite").saveAsTable("vehicle_insurance.gold.claims_settlement_kpi")

payment_kpi.write.format("delta").mode("overwrite").saveAsTable("vehicle_insurance.gold.payment_kpi")

high_risk_customers.write.format("delta").mode("overwrite").saveAsTable("vehicle_insurance.gold.high_risk_customers")

policy_kpi.write.format("delta").mode("overwrite").saveAsTable("vehicle_insurance.gold.policy_kpi")

vehicle_kpi.write.format("delta").mode("overwrite").saveAsTable("vehicle_insurance.gold.vehicle_kpi")

fnol_kpi.write.format("delta").mode("overwrite").saveAsTable("vehicle_insurance.gold.fnol_kpi")

incident_kpi.write.format("delta").mode("overwrite").saveAsTable("vehicle_insurance.gold.incident_kpi")

repeat_claims.write.format("delta").mode("overwrite").saveAsTable("vehicle_insurance.gold.repeat_claims")

multiple_fnol.write.format("delta").mode("overwrite").saveAsTable("vehicle_insurance.gold.multiple_fnol")

high_claim_outliers.write.format("delta").mode("overwrite").saveAsTable("vehicle_insurance.gold.high_claim_outliers")